## Cleaning

### Data Import

In [1]:
import polars as pl
import altair as alt
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # override=True ensures .env values take precedence over system env vars

USER = os.getenv("USER")
PASSWORD = os.getenv("PW")
HOST = os.getenv("HOST")
DATABASE = os.getenv("DB_NAME")

uri = f"postgresql://{USER}:{PASSWORD}@{HOST}:5432/{DATABASE}"

query = "SELECT * FROM daan_822.article_detailed"

df = pl.read_database_uri(query=query, uri=uri)

In [2]:
df.describe()

statistic,pmid,doi,article_type,article_title,article_subject,authors,pub_date,keywords,reference_count,license_type,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available
str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,f64,str,f64
"""count""","""7178""","""7167""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""0""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""",7178.0
"""null_count""","""0""","""11""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""7178""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,null,null,null,null,null,53.387852,null,null,null,null,null,null,null,0.657147,null,null,null,null,0.395653,null,0.033853
"""std""",null,null,null,null,null,null,null,null,48.141303,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""",""" Helicobacter pylori Screeni…","""""","""[]""","""2024-10-08""","""["" Artificial intelligence"",""S…",0.0,null,"""AACE Endocrinology and Diabete…","""""","""""","""""","""""","""[""*Brain Trauma Foundation, Pa…",0.0,"""0""","""0""","""["" The APC was funded by Wenta…","""["" Data are available on reque…",0.0,"""[""A GitHub repository containi…",0.0
"""25%""",null,null,null,null,null,null,null,null,31.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,43.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,59.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""Zoster""","""[{""given_names"":""Šimon"",""is_co…","""2026-5-12""","""[]""",1194.0,null,"""npj Metabolic Health and Disea…","""ecancer Global Foundation""","""©Zongqi Xia, Prerna Chikersal,…","""2026""","""​​BackgroundOsteoporosis, a pr…","""[]""",1.0,"""9""","""9""","""[]""","""[]""",1.0,"""[]""",1.0


### Cleaning
- Strip leading/trailing whitespace from all string columns
- Replace empty strings with null across all string columns
- Cast copyright_year, figure_count, table_count from Text to Int16
- Parse pub_date string to Date type with multi-format fallback (YYYY-MM-DD, YYYY-MM, YYYY)
- Drop license_type column (entirely null)
- Lowercase keywords, journal_title, publisher_name, article_type, article_subject
- Parse authors JSON string into a typed struct list
- Parse keywords JSON string into a string list
- Detect remaining JSON array columns and convert to comma-separated strings
- Explode authors into df_authors; strip whitespace, null out empty names, validate ORCID format (^\d{4}-\d{4}-\d{4}-\d{3}[\dX]$), build author_full_name
- Explode keywords into df_keyword

In [3]:
# cleaning whitespace and replacing blank with true nulls
df_cleaning = df.with_columns(pl.selectors.string().str.strip_chars().replace("", None))

# converting string type to int
df_cleaning = df_cleaning.with_columns(
    pl.selectors.by_name("copyright_year", "figure_count", "table_count").cast(pl.Int16)
)

# converting pub_date to date
df_cleaning = df_cleaning.with_columns(
    pl.coalesce(
        [
            pl.col("pub_date").str.to_date("%Y-%m-%d", strict=False),
            pl.col("pub_date").str.to_date("%Y-%m", strict=False),
            pl.col("pub_date").str.to_date("%Y", strict=False),
        ]
    )
)

# dropping column as it is null
df_cleaning = df_cleaning.drop("license_type")

# making columns lower case
df_cleaning = df_cleaning.with_columns(
    pl.selectors.by_name("keywords", "journal_title", "publisher_name", "article_type", "article_subject").str.to_lowercase()
)

In [4]:
df_cleaning.describe()

statistic,pmid,doi,article_type,article_title,article_subject,authors,pub_date,keywords,reference_count,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available
str,str,str,str,str,str,str,str,str,f64,str,str,str,f64,str,str,f64,f64,f64,str,str,f64,str,f64
"""count""","""7178""","""7167""","""7178""","""7178""","""7140""","""7178""","""7156""","""7178""",7178.0,"""7178""","""6518""","""6871""",6105.0,"""7117""","""7178""",7178.0,7178.0,7178.0,"""7178""","""7178""",7178.0,"""7178""",7178.0
"""null_count""","""0""","""11""","""0""","""0""","""38""","""0""","""22""","""0""",0.0,"""0""","""660""","""307""",1073.0,"""61""","""0""",0.0,0.0,0.0,"""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,null,null,null,"""2025-07-02 23:04:15.561766""",null,53.387852,null,null,null,2024.984767,null,null,0.657147,3.942045,3.011702,null,null,0.395653,null,0.033853
"""std""",null,null,null,null,null,null,null,null,48.141303,null,null,null,0.150133,null,null,null,7.704139,2.888119,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""","""10 Practical Considerations fo…","""10""","""[]""","""2024-03-20""","""["" artificial intelligence"",""s…",0.0,"""aace endocrinology and diabete…","""a k peters/crc press""","""(c) 2025 The authors; licensee…",2023.0,"""(1) Background: Alveolar bone …","""[""*Brain Trauma Foundation, Pa…",0.0,0.0,0.0,"""["" The APC was funded by Wenta…","""["" Data are available on reque…",0.0,"""[""A GitHub repository containi…",0.0
"""25%""",null,null,null,null,null,null,"""2025-04-04""",null,31.0,null,null,null,2025.0,null,null,null,2.0,2.0,null,null,null,null,null
"""50%""",null,null,null,null,null,null,"""2025-07-07""",null,43.0,null,null,null,2025.0,null,null,null,3.0,3.0,null,null,null,null,null
"""75%""",null,null,null,null,null,null,"""2025-10-06""",null,59.0,null,null,null,2025.0,null,null,null,5.0,4.0,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""zoster""","""[{""given_names"":""Šimon"",""is_co…","""2026-05-12""","""[]""",1194.0,"""yonsei medical journal""","""zenodo""","""©Zongqi Xia, Prerna Chikersal,…",2026.0,"""​​BackgroundOsteoporosis, a pr…","""[]""",1.0,487.0,77.0,"""[]""","""[]""",1.0,"""[]""",1.0


In [5]:
# labelling field names in the json construct for authors
dtypes = pl.List(
    pl.Struct(
        [
            pl.Field("given_names", pl.String),
            pl.Field("is_corresponding", pl.Boolean),
            pl.Field("orcid", pl.String),
            pl.Field("surname", pl.String),
        ]
    )
)

# parse JSON columns without exploding to avoid author x keyword cartesian product
df_parsed = df_cleaning.with_columns(
    pl.col("authors").str.json_decode(dtypes).alias("authors_struct"),
    pl.col("keywords").str.json_decode(pl.List(pl.String)).alias("keywords_list"),
).drop("authors", "keywords")

# identify columns that are json formatted in order to clean to be comma separated
json_array_cols = [
    col
    for col, dtype in df_parsed.schema.items()
    if dtype == pl.String
    and df_parsed.select(pl.col(col).drop_nulls().str.starts_with("[").all()).item()
]

# clean up JSON array columns to comma-separated strings
df_base = df_parsed.with_columns(
    [
        pl.col(col).str.json_decode(pl.List(pl.String)).list.join(", ").alias(col)
        for col in json_array_cols
    ]
)

# separate exploded views to avoid author x keyword cartesian product
df_authors = (
    df_base.explode("authors_struct")
    .unnest("authors_struct")
    .with_columns(
        # strip whitespace from name fields
        pl.col("given_names").str.strip_chars(),
        pl.col("surname").str.strip_chars(),
        pl.col("orcid").str.strip_chars(),
    )
    .with_columns(
        # replace empty strings with null
        pl.col("given_names").replace("", None),
        pl.col("surname").replace("", None),
        pl.col("orcid").replace("", None),
    )
    .with_columns(
        # validate ORCID format — set invalid values to null
        pl.when(pl.col("orcid").str.contains(r"^\d{4}-\d{4}-\d{4}-\d{3}[\dX]$"))
        .then(pl.col("orcid"))
        .otherwise(None)
        .alias("orcid")
    )
    .with_columns(
        (pl.col("given_names").fill_null("") + " " + pl.col("surname").fill_null(""))
        .str.strip_chars()
        .replace("", None)
        .alias("author_full_name")
    )
)

df_keywords = df_base.explode("keywords_list").rename({"keywords_list": "keyword"})

Run lemming on keywords to try to resolve duplicates

In [6]:
import spacy

In [7]:
nlp = spacy.load("en_core_web_sm")


def normalize_keyword(keyword: str) -> str:
    doc = nlp(keyword.lower())
    return " ".join(
        tok.lemma_
        for tok in doc
        if not tok.is_stop and tok.is_alpha
    )

In [8]:
df_keywords = df_keywords.with_columns(
    pl.col("keyword")
      .map_elements(normalize_keyword)
      .alias("keyword_lemma")
)

In [9]:
df_keywords.select("pmid", "keyword", "keyword_lemma").limit(10)

pmid,keyword,keyword_lemma
str,str,str
"""41215574""","""electronic health records""","""electronic health record"""
"""41215574""","""diagnostic error""","""diagnostic error"""
"""41215574""","""patient safety event reports""","""patient safety event report"""
"""41249591""","""respiratory tract diseases""","""respiratory tract disease"""
"""41249591""","""paediatrics""","""paediatric"""
"""40307592""","""dual-energy ct""","""dual energy ct"""
"""40307592""","""virtual monochromatic images""","""virtual monochromatic image"""
"""40307592""","""deep learning image reconstruc…","""deep learn image reconstructio…"
"""40307592""","""pancreatic ductal adenocarcino…","""pancreatic ductal adenocarcino…"


In [10]:
df_keywords.describe()

statistic,pmid,doi,article_type,article_title,article_subject,pub_date,reference_count,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available,authors_struct,keyword,keyword_lemma
str,str,str,str,str,str,str,f64,str,str,str,f64,str,str,f64,f64,f64,str,str,f64,str,f64,f64,str,str
"""count""","""35969""","""35924""","""35969""","""35969""","""35756""","""35889""",35969.0,"""35969""","""33344""","""34690""",31240.0,"""35787""","""35969""",35969.0,35969.0,35969.0,"""35969""","""35969""",35969.0,"""35969""",35969.0,35969.0,"""34957""","""34957"""
"""null_count""","""0""","""45""","""0""","""0""","""213""","""80""",0.0,"""0""","""2625""","""1279""",4729.0,"""182""","""0""",0.0,0.0,0.0,"""0""","""0""",0.0,"""0""",0.0,0.0,"""1012""","""1012"""
"""mean""",null,null,null,null,null,"""2025-07-03 01:36:20.222352""",55.951236,null,null,null,2024.982618,null,null,0.644861,3.873502,3.159915,null,null,0.372682,null,0.022881,null,null,null
"""std""",null,null,null,null,null,null,56.69863,null,null,null,0.155314,null,null,null,4.423236,3.119667,null,null,null,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""","""10 Practical Considerations fo…","""10""","""2024-03-20""",0.0,"""aace endocrinology and diabete…","""a k peters/crc press""","""(c) 2025 The authors; licensee…",2023.0,"""(1) Background: Alveolar bone …","""""",0.0,0.0,0.0,"""""","""""",0.0,"""""",0.0,null,""" borrelia burgdorferi sensu …",""""""
"""25%""",null,null,null,null,null,"""2025-04-02""",32.0,null,null,null,2025.0,null,null,null,2.0,2.0,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,"""2025-07-07""",44.0,null,null,null,2025.0,null,null,null,3.0,3.0,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,"""2025-10-06""",61.0,null,null,null,2025.0,null,null,null,5.0,4.0,null,null,null,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""zoster""","""2026-05-12""",1194.0,"""yonsei medical journal""","""zenodo""","""©Zongqi Xia, Prerna Chikersal,…",2026.0,"""​​BackgroundOsteoporosis, a pr…","""‡Department of Neurosurgery, V…",1.0,487.0,77.0,"""贵州省第八批高层次创新型“百”层次人才""","""“For ethical and administrativ…",1.0,"""eHarmonize is an open-source p…",1.0,null,""" propofol""","""طب الأعصاب"""


#### Missingness Handler

In [11]:
total_rows = df_base.height

cols_with_null_info = [
    (col, df_base.select(pl.col(col).null_count()).item(),
     round(df_base.select(pl.col(col).null_count()).item() / total_rows * 100, 2), df_base.schema[col])
    for col in df_base.columns
    if df_base.select(pl.col(col).null_count()).item() > 0
]

# how many rows are missing copyright_year but have a pub_date?
missing_cy = df_base.filter(pl.col("copyright_year").is_null())

print(f"Missing copyright_year: {missing_cy.height}")
print(f"  - with pub_date available: {missing_cy.filter(pl.col('pub_date').is_not_null()).height}")
print(f"  - without pub_date:        {missing_cy.filter(pl.col('pub_date').is_null()).height}")

df_base = df_cleaning.with_columns(
    pl.when(pl.col("copyright_year").is_null())
    .then(pl.col("pub_date").dt.year().cast(pl.Int16))
    .otherwise(pl.col("copyright_year"))
    .alias("copyright_year")
)

Missing copyright_year: 1073
  - with pub_date available: 1067
  - without pub_date:        6


#### Copyright Year validation

In [12]:
# extracts year from the copyright statement
df_base = df_base.with_columns(
    pl.col("copyright_statement")
    .str.extract(r"((?:19|20)\d{2})")
    .cast(pl.Int16, strict=False)
    .alias("year_from_statement")
)

# compares the result
df_base = df_base.with_columns(
    (pl.col("copyright_year") == pl.col("year_from_statement")).alias("copyright_year_match")
)

# if the year extracted is not null, it replaces the copyright year
df_base = df_base.with_columns(
    pl.when(pl.col("year_from_statement").is_not_null())
    .then(pl.col("year_from_statement"))
    .otherwise(pl.col("copyright_year"))
    .alias("copyright_year")
)

# there are only 25 rows that did not match when compared
copyright_year_mismatch_df = df_base.filter(pl.col("copyright_year_match") == False)

df_base = df_base.with_columns(
    pl.col("pub_date").dt.year().cast(pl.Int16).alias("pub_year")
).with_columns(
    (pl.col("copyright_year") == pl.col("pub_year")).alias("copyright_eq_pub")
)

### Outliers

Create embeddings for article titles

In [33]:
from sentence_transformers import SentenceTransformer

In [34]:
model = SentenceTransformer("pritamdeka/S-Scibert-snli-multinli-stsb")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-Scibert-snli-multinli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
df_lazy = df_base.lazy()

article_titles = df_lazy.select("article_title").collect()["article_title"].to_list()


embeddings = model.encode(
    article_titles,
    normalize_embeddings=True
)


In [39]:

df_lazy = df_lazy.with_columns(
    pl.Series("article_title_embedding", embeddings.tolist())
)

df_base = df_lazy.collect()
df_base.schema

Schema([('pmid', String),
        ('doi', String),
        ('article_type', String),
        ('article_title', String),
        ('article_subject', String),
        ('authors', String),
        ('pub_date', Date),
        ('keywords', String),
        ('reference_count', Int32),
        ('journal_title', String),
        ('publisher_name', String),
        ('copyright_statement', String),
        ('copyright_year', Int16),
        ('abstract', String),
        ('affiliations', String),
        ('has_supplemental', Boolean),
        ('figure_count', Int16),
        ('table_count', Int16),
        ('funding', String),
        ('data_available_details', String),
        ('data_available', Boolean),
        ('code_available_details', String),
        ('code_available', Boolean),
        ('year_from_statement', Int16),
        ('copyright_year_match', Boolean),
        ('pub_year', Int16),
        ('copyright_eq_pub', Boolean),
        ('article_title_embedding', List(Float64))])

Now use k-nearest neighbor to detect articles that are outliers based on their titles

In [40]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [41]:
X = np.vstack(df_base["article_title_embedding"].to_list())

k = 10

nn = NearestNeighbors(n_neighbors=k + 1
                      , metric="cosine")
nn.fit(X)


distances, _ = nn.kneighbors(X)
outlier_score = distances[:, 1:].mean(axis=1)

df_base = df_base.with_columns(
    pl.Series("outlier_score", outlier_score)
)

Use threshold to detect outliers

In [50]:

threshold = df_base.select(
    pl.col("outlier_score").quantile(0.95)
).item()

df_base = df_base.with_columns(
    (pl.col("outlier_score") > threshold)
    .alias("is_outlier")
)
len(df_base.select("article_title", "article_type", "outlier_score", "is_outlier").filter(pl.col("is_outlier")))
df_base.select("article_title", "article_type", "outlier_score", "is_outlier").filter(pl.col("is_outlier"))

article_title,article_type,outlier_score,is_outlier
str,str,f64,bool
"""Validation of the French Trans…","""letter""",0.5382,true
"""A case study on SSD to SAD lin…","""other""",0.554199,true
"""Turbulence drives arteriovenou…","""research-article""",0.551618,true
"""Association of CSF Kappa-Free …","""research-article""",0.55905,true
"""The Impact of Lunar Phases Dur…","""research-article""",0.539445,true
…,…,…,…
"""Generation Victoria (GenV): pr…","""research-article""",0.578996,true
"""What are the hidden shortcomin…","""research-article""",0.560864,true
"""Behçet’s disease unraveled: In…","""research-article""",0.57245,true


Filter out the outlier titles

In [51]:
df_base = df_base.filter(~pl.col("is_outlier")).drop(["article_title_embedding", "outlier_score", "is_outlier"])

## Analysis

In [52]:
df_base.schema

Schema([('pmid', String),
        ('doi', String),
        ('article_type', String),
        ('article_title', String),
        ('article_subject', String),
        ('authors', String),
        ('pub_date', Date),
        ('keywords', String),
        ('reference_count', Int32),
        ('journal_title', String),
        ('publisher_name', String),
        ('copyright_statement', String),
        ('copyright_year', Int16),
        ('abstract', String),
        ('affiliations', String),
        ('has_supplemental', Boolean),
        ('figure_count', Int16),
        ('table_count', Int16),
        ('funding', String),
        ('data_available_details', String),
        ('data_available', Boolean),
        ('code_available_details', String),
        ('code_available', Boolean),
        ('year_from_statement', Int16),
        ('copyright_year_match', Boolean),
        ('pub_year', Int16),
        ('copyright_eq_pub', Boolean)])

In [14]:
df.describe()

statistic,pmid,doi,article_type,article_title,article_subject,authors,pub_date,keywords,reference_count,license_type,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available
str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,f64,str,f64
"""count""","""7178""","""7167""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""0""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""",7178.0
"""null_count""","""0""","""11""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""7178""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,null,null,null,null,null,53.387852,null,null,null,null,null,null,null,0.657147,null,null,null,null,0.395653,null,0.033853
"""std""",null,null,null,null,null,null,null,null,48.141303,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""",""" Helicobacter pylori Screeni…","""""","""[]""","""2024-10-08""","""["" Artificial intelligence"",""S…",0.0,null,"""AACE Endocrinology and Diabete…","""""","""""","""""","""""","""[""*Brain Trauma Foundation, Pa…",0.0,"""0""","""0""","""["" The APC was funded by Wenta…","""["" Data are available on reque…",0.0,"""[""A GitHub repository containi…",0.0
"""25%""",null,null,null,null,null,null,null,null,31.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,43.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,59.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""Zoster""","""[{""given_names"":""Šimon"",""is_co…","""2026-5-12""","""[]""",1194.0,null,"""npj Metabolic Health and Disea…","""ecancer Global Foundation""","""©Zongqi Xia, Prerna Chikersal,…","""2026""","""​​BackgroundOsteoporosis, a pr…","""[]""",1.0,"""9""","""9""","""[]""","""[]""",1.0,"""[]""",1.0


In [15]:
# creating a summary dataframe of common metrics
summary_df = pl.DataFrame(
    {
        "Metric": [
            "Total Articles",
            "Unique Authors",
            "Unique Keywords",
            "Avg Authors per Paper",
            "Avg Keywords per Paper",
        ],
        "Value": [
            df_base.select(pl.col("pmid").n_unique()).item(),
            df_authors.select(pl.col("author_full_name").n_unique()).item(),
            df_keywords.select(pl.col("keyword").n_unique()).item(),
            df_authors.group_by("pmid")
            .agg(pl.col("author_full_name").n_unique().alias("n_authors"))
            .select(pl.col("n_authors").mean().round(2))
            .item(),
            df_keywords.group_by("pmid")
            .agg(pl.col("keyword").n_unique().alias("n_keywords"))
            .select(pl.col("n_keywords").mean().round(2))
            .item(),
        ],
    },
    strict=False,
)

summary_df

Metric,Value
str,f64
"""Total Articles""",7178.0
"""Unique Authors""",72961.0
"""Unique Keywords""",16177.0
"""Avg Authors per Paper""",11.47
"""Avg Keywords per Paper""",4.99


#### Building Visuals

In [16]:
data_avail = df_base.group_by("data_available").agg(pl.len().alias("count"))

data_avail

data_available,count
bool,u32
false,4338
true,2840


In [17]:
pub_year_counts = (
    df_base.with_columns(pl.col("pub_date").dt.year().alias("pub_year"))
    .group_by("pub_year")
    .agg(pl.col("pmid").n_unique().alias("num_articles"))
    .sort("pub_year")
)

pub_year_counts

pub_year,num_articles
i32,u32
null,22
2024,91
2025,7058
2026,7


In [18]:
pub_month_counts = (
    df_base
    .with_columns(
        pl.col("pub_date")
          .dt.truncate("1mo")
          .alias("pub_month")
    )
    .group_by("pub_month")
    .agg(pl.col("pmid").n_unique().alias("num_articles"))
    .sort("pub_month")
)

pub_month_counts

pub_month,num_articles
date,u32
null,22
2024-03-01,1
2024-04-01,1
2024-05-01,2
2024-07-01,7
…,…
2025-11-01,643
2025-12-01,566
2026-01-01,4


In [19]:
top_keywords = (
    df_keywords
    .filter(pl.col("keyword").is_not_null())
    .group_by("keyword")
    .agg(pl.col("pmid").n_unique().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

top_keywords

keyword,count
str,u32
"""machine learning""",527
"""artificial intelligence""",425
"""epidemiology""",201
"""electronic health records""",181
"""mortality""",181
…,…
"""sepsis""",113
"""breast cancer""",107
"""digital health""",106


In [20]:
top_authors = (
    df_authors
    .filter(pl.col("author_full_name").is_not_null())
    .group_by("author_full_name")
    .agg(pl.col("pmid").n_unique().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

top_authors

author_full_name,count
str,u32
"""Kiyohide Fushimi""",45
"""Hideo Yasunaga""",24
"""Hiroki Matsui""",21
"""Jiang Bian""",21
"""Rohan Khera""",16
…,…
"""Wei Wang""",13
"""Phichayut Phinyo""",13
"""Xin Wang""",13


#### Visualizing

In [21]:
(
    alt.Chart(data_avail)
    .mark_bar()
    .encode(
        x=alt.X("data_available:O", title=""),
        y=alt.Y("count:Q", title="Number of Articles"),
    )
    .properties(title="Data Availability")
)

alt.Chart(...)

In [22]:
(
    alt.Chart(pub_year_counts)
    .mark_bar()
    .encode(
        x=alt.X("pub_year:O", title="Publication Year"),
        y=alt.Y("num_articles:Q", title="Number of Articles"),
    )
    .properties(title="Publications per Year")
)

alt.Chart(...)

In [23]:
(
    alt.Chart(pub_month_counts)
    .mark_line(point=True)
    .encode(
        x=alt.X("pub_month:T", title="Publication Date"),
        y=alt.Y("num_articles:Q", title="Number of Articles")
    )
    .properties(title="Publications Over Time")
)

alt.Chart(...)

In [24]:
(
    alt.Chart(top_keywords)
    .mark_bar()
    .encode(
        x=alt.X(
            "keyword:O",
            title="Keyword",
            sort=alt.SortField("count", order="descending"),
        ),
        y=alt.Y("count:Q", title="Count"),
    )
    .properties(title="Top 15 Keywords")
)

alt.Chart(...)

In [25]:
(
    alt.Chart(top_authors)
    .mark_bar()
    .encode(
        x=alt.X(
            "author_full_name:O",
            title="Author",
            sort=alt.SortField("count", order="descending"),
        ),
        y=alt.Y("count:Q", title="Number of Articles"),
    )
    .properties(title="Top 15 Authors")
)

alt.Chart(...)

## Export dataframes to postgres

In [26]:
from database import engine

In [27]:
df_keywords.schema

Schema([('pmid', String),
        ('doi', String),
        ('article_type', String),
        ('article_title', String),
        ('article_subject', String),
        ('pub_date', Date),
        ('reference_count', Int32),
        ('journal_title', String),
        ('publisher_name', String),
        ('copyright_statement', String),
        ('copyright_year', Int16),
        ('abstract', String),
        ('affiliations', String),
        ('has_supplemental', Boolean),
        ('figure_count', Int16),
        ('table_count', Int16),
        ('funding', String),
        ('data_available_details', String),
        ('data_available', Boolean),
        ('code_available_details', String),
        ('code_available', Boolean),
        ('authors_struct',
         List(Struct({'given_names': String, 'is_corresponding': Boolean, 'orcid': String, 'surname': String}))),
        ('keyword', String),
        ('keyword_lemma', String)])

In [28]:
df_keywords.select(pl.col("pmid"), pl.col("keyword"), pl.col("keyword_lemma")).write_database(
    table_name="keyword",
    connection=engine,
    if_table_exists="replace"
)

969

In [29]:
df_authors.schema

Schema([('pmid', String),
        ('doi', String),
        ('article_type', String),
        ('article_title', String),
        ('article_subject', String),
        ('pub_date', Date),
        ('reference_count', Int32),
        ('journal_title', String),
        ('publisher_name', String),
        ('copyright_statement', String),
        ('copyright_year', Int16),
        ('abstract', String),
        ('affiliations', String),
        ('has_supplemental', Boolean),
        ('figure_count', Int16),
        ('table_count', Int16),
        ('funding', String),
        ('data_available_details', String),
        ('data_available', Boolean),
        ('code_available_details', String),
        ('code_available', Boolean),
        ('given_names', String),
        ('is_corresponding', Boolean),
        ('orcid', String),
        ('surname', String),
        ('keywords_list', List(String)),
        ('author_full_name', String)])

In [30]:
df_authors.select("pmid", "given_names", "is_corresponding", "orcid", "surname", "author_full_name").write_database(
    table_name="authors",
    connection=engine,
    if_table_exists="replace"
)

109

In [31]:
df_base.schema

Schema([('pmid', String),
        ('doi', String),
        ('article_type', String),
        ('article_title', String),
        ('article_subject', String),
        ('authors', String),
        ('pub_date', Date),
        ('keywords', String),
        ('reference_count', Int32),
        ('journal_title', String),
        ('publisher_name', String),
        ('copyright_statement', String),
        ('copyright_year', Int16),
        ('abstract', String),
        ('affiliations', String),
        ('has_supplemental', Boolean),
        ('figure_count', Int16),
        ('table_count', Int16),
        ('funding', String),
        ('data_available_details', String),
        ('data_available', Boolean),
        ('code_available_details', String),
        ('code_available', Boolean),
        ('year_from_statement', Int16),
        ('copyright_year_match', Boolean),
        ('pub_year', Int16),
        ('copyright_eq_pub', Boolean)])

In [32]:
df_base.write_database(
    table_name="article_clean",
    connection=engine,
    if_table_exists="replace"
)

178